In [1]:
import glob
import json
import pandas as pd
from tqdm import tqdm

In [2]:
result_paths = glob.glob('./Result/analysis_reports_labeled/*')
result_paths = sorted(result_paths, key=lambda x: x.split('/')[-1])

In [ ]:
df_snippet_score = pd.DataFrame(columns=['company_name', 'date', 'item_label', 'number_of_evidence', 'evidence_no', 'final_snippet_score'])

for f_p in tqdm(result_paths):
    with open(f_p, 'r', encoding='utf-8') as f:
        result = json.load(f)[0]
    count_1a = 0
    count_7 = 0
    company_name = f_p.split('/')[-1].split('_')[0]
    date = f_p.split('/')[-1].split('_')[1].split('.')[0]
    evidence_list = result['evidence_list']
    number_of_evidence = len(evidence_list)
    for i, evidence in enumerate(evidence_list):
        evidence_no = i + 1
        item_label = evidence['item_label']
        final_snippet_score = evidence['score_calculation']['final_snippet_score']
        df_snippet_score.loc[len(df_snippet_score)] = [company_name, date, item_label, number_of_evidence, evidence_no, final_snippet_score]

        if item_label == 'item_1a':
            count_1a += 1
        elif item_label == 'item_7':
            count_7 += 1
        else:
            count_1a += 1
            count_7 += 1
    # if count_1a == 0 and count_7 == 0:
    #     print(f"Warning: {company_name} on {date} has no item_1a or item_7 evidence.")

# df_snippet_score.to_csv('./Result/llm_result/snippet_score.csv', index=False, encoding='utf-8-sig')
df_snippet_score

 17%|█▋        | 300/1728 [00:00<00:03, 470.09it/s]

 37%|███▋      | 645/1728 [00:01<00:03, 358.59it/s]

 53%|█████▎    | 919/1728 [00:02<00:02, 309.54it/s]

 70%|███████   | 1213/1728 [00:03<00:01, 275.70it/s]

 83%|████████▎ | 1428/1728 [00:04<00:01, 237.71it/s]

 85%|████████▌ | 1476/1728 [00:04<00:01, 230.01it/s]

 96%|█████████▌| 1654/1728 [00:05<00:00, 248.52it/s]

 99%|█████████▉| 1707/1728 [00:05<00:00, 245.40it/s]

100%|██████████| 1728/1728 [00:05<00:00, 304.09it/s]


,company_name,date,item_label,number_of_evidence,evidence_no,final_snippet_score
0,AAPL,2021-10-29,item_1a,18,1,6
1,AAPL,2021-10-29,"item_1a, item_7",18,2,3
2,AAPL,2021-10-29,"item_1a, item_7",18,3,3
3,AAPL,2021-10-29,"item_1a, item_7",18,4,3
4,AAPL,2021-10-29,item_1a,18,5,5
...,...,...,...,...,...,...
16436,ZTS,2024-02-13,item_1a,5,1,3
16437,ZTS,2024-02-13,item_1a,5,2,2
16438,ZTS,2024-02-13,item_1a,5,3,3
16439,ZTS,2024-02-13,"item_1a, item_7",5,4,4


In [4]:
df_snippet_score['item_1a'] = [1 if 'item_1a' in label else 0 for label in df_snippet_score['item_label']]
df_snippet_score['item_7'] = [1 if 'item_7' in label else 0 for label in df_snippet_score['item_label']]
df_snippet_score

,company_name,date,item_label,number_of_evidence,evidence_no,final_snippet_score,item_1a,item_7
0,AAPL,2021-10-29,item_1a,18,1,6,1,0
1,AAPL,2021-10-29,"item_1a, item_7",18,2,3,1,1
2,AAPL,2021-10-29,"item_1a, item_7",18,3,3,1,1
3,AAPL,2021-10-29,"item_1a, item_7",18,4,3,1,1
4,AAPL,2021-10-29,item_1a,18,5,5,1,0
...,...,...,...,...,...,...,...,...
16436,ZTS,2024-02-13,item_1a,5,1,3,1,0
16437,ZTS,2024-02-13,item_1a,5,2,2,1,0
16438,ZTS,2024-02-13,item_1a,5,3,3,1,0
16439,ZTS,2024-02-13,"item_1a, item_7",5,4,4,1,1


In [18]:
df_item_1a = df_snippet_score[df_snippet_score['item_1a'] == 1].copy()
df_item_1a['item_label'] = ['item_1a'] * len(df_item_1a)
df_item_1a = df_item_1a.drop(columns=['item_1a', 'item_7'])

df_item_7 = df_snippet_score[df_snippet_score['item_7'] == 1].copy()
df_item_7['item_label'] = ['item_7'] * len(df_item_7)
df_item_7 = df_item_7.drop(columns=['item_1a', 'item_7'])
df_item_7

,company_name,date,item_label,number_of_evidence,evidence_no,final_snippet_score
1,AAPL,2021-10-29,item_7,18,2,3
2,AAPL,2021-10-29,item_7,18,3,3
3,AAPL,2021-10-29,item_7,18,4,3
5,AAPL,2021-10-29,item_7,18,6,7
6,AAPL,2021-10-29,item_7,18,7,3
...,...,...,...,...,...,...
16418,ZTS,2023-02-14,item_7,21,4,5
16423,ZTS,2023-02-14,item_7,21,9,9
16424,ZTS,2023-02-14,item_7,21,10,3
16426,ZTS,2023-02-14,item_7,21,12,1


In [29]:
df_snippet_all = pd.concat([df_item_1a, df_item_7], ignore_index=True)
df_snippet_all.to_csv('./Result/llm_result/snippet_score_labeled.csv', index=False, encoding='utf-8-sig')

In [55]:
df_snippet_count = df_snippet_all.groupby(['company_name', 'date', 'item_label']).agg(item_snippets_score=('final_snippet_score', 'sum'),
                                                                   snippets_count=('final_snippet_score', 'count'),
                                                                   max_snippet_score=('final_snippet_score', 'max'),
                                                                   min_snippet_score=('final_snippet_score', 'min'),
                                                                   mean_snippet_score=('final_snippet_score', 'mean'),
                                                                   std_snippet_score=('final_snippet_score', 'std')).reset_index()
df_snippet_count
df_snippet_count.to_csv('./Result/llm_result/snippet_score_item_count.csv', index=False, encoding='utf-8-sig')